# 必要な情報を聞き出しながら予約を受け付ける (スロットフィリング)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/22_dialogue/slot_filling.ipynb)

NLP クックブック 22-04 ／ 対話システム

「日時」「人数」「名前」のように、タスクに必要な項目が揃うまで質問を続ける対話を作りたい。一度に複数の情報を言われても取りこぼさないようにしたい。

In [1]:
!pip install -q ja-timex

In [2]:
import re
import unicodedata
import pendulum
from ja_timex import TimexParser

TODAY = pendulum.datetime(2026, 11, 14, tz="Asia/Tokyo")      # 相対的な日付の基準 (実際は pendulum.now())
WEEKDAYS = "月火水木金土日"

def extract(text):
    """発話から、日付・時刻・人数・名前を取り出す"""
    text = unicodedata.normalize("NFKC", text)
    slots = {}
    if m := re.search(r"(\d+)\s*(?:人|名)|(一人|ひとり|二人|ふたり)", text):
        slots["人数"] = int(m.group(1)) if m.group(1) else {"一人": 1, "ひとり": 1, "二人": 2, "ふたり": 2}[m.group(2)]
    if m := re.search(r"(?:名前は|予約名は)?\s*([一-龥ぁ-んァ-ヶー]{2,6})(?:です|と申します|でお願いします)$", text.strip("。 ")):
        if not re.search(r"\d|人|名|時|日|曜", m.group(1)):
            slots["名前"] = m.group(1)
    for t in TimexParser(reference=TODAY).parse(text):
        if t.type == "TIME" and (m := re.match(r"T(\d\d)-(\d\d|XX)", t.value)):
            hour = int(m.group(1))
            if hour < 11 and "午前" not in text and "朝" not in text:
                hour += 12                                       # 飲食店なので「7 時」は 19 時と解釈する
            slots["時刻"] = f"{hour:02d}:{'00' if m.group(2) == 'XX' else m.group(2)}"
        elif t.type == "DATE" and (m := re.match(r"(\d{4}|XXXX)-(\d\d)-(\d\d)", t.value)):
            slots["日付"] = f"{TODAY.year if m.group(1) == 'XXXX' else m.group(1)}-{m.group(2)}-{m.group(3)}"
        elif t.type == "DATE" and (m := re.match(r"XXXX-WXX-(\d)", t.value)):
            days = (int(m.group(1)) - 1 - TODAY.weekday()) % 7 or 7     # 次に来るその曜日
            slots["日付"] = TODAY.add(days=days + (7 if "来週" in text else 0)).to_date_string()
        elif t.type == "DURATION" and t.mod == "AFTER" and (m := re.match(r"P(\d+)D", t.value)):
            slots["日付"] = TODAY.add(days=int(m.group(1))).to_date_string()     # 明日、明後日
    if "今日" in text or "本日" in text:
        slots["日付"] = TODAY.to_date_string()
    return slots

for text in ["明日の夜7時に4人で", "来週の金曜日、19時半から2名", "11月20日の18時でお願いします", "山田です", "3人です"]:
    print(text, "→", extract(text))

明日の夜7時に4人で → {'人数': 4, '日付': '2026-11-15', '時刻': '19:00'}
来週の金曜日、19時半から2名 → {'人数': 2, '日付': '2026-11-27', '時刻': '19:30'}
11月20日の18時でお願いします → {'日付': '2026-11-20', '時刻': '18:00'}
山田です → {'名前': '山田'}
3人です → {'人数': 3}


In [3]:
QUESTIONS = {"日付": "ご希望の日にちを教えてください。", "時刻": "何時からにしますか？", "人数": "何名様ですか？", "名前": "ご予約のお名前をお願いします。"}

class Reservation:
    def __init__(self):
        self.slots = {k: None for k in QUESTIONS}

    def reply(self, text):
        for k, v in extract(text).items():        # 言われた情報はすべて取り込む (聞いていない項目でも)
            self.slots[k] = v
        missing = [k for k, v in self.slots.items() if v is None]
        if missing:
            return QUESTIONS[missing[0]]           # 足りない項目を 1 つずつ聞く
        s = self.slots
        d = pendulum.parse(s["日付"])
        return f"{d.month} 月 {d.day} 日 ({WEEKDAYS[d.weekday()]}) {s['時刻']} から {s['人数']} 名様、{s['名前']}様でご予約を承りました。"

for turns in [["予約をお願いします", "明日の夜7時に4人で", "山田です"], ["来週の金曜日に2名で予約したい", "19時半で", "佐藤と申します"]]:
    bot = Reservation()
    for user in turns:
        print(f"客: {user}\n店: {bot.reply(user)}")
    print(bot.slots, "\n")

客: 予約をお願いします
店: ご希望の日にちを教えてください。
客: 明日の夜7時に4人で
店: ご予約のお名前をお願いします。
客: 山田です
店: 11 月 15 日 (日) 19:00 から 4 名様、山田様でご予約を承りました。
{'日付': '2026-11-15', '時刻': '19:00', '人数': 4, '名前': '山田'} 

客: 来週の金曜日に2名で予約したい
店: 何時からにしますか？
客: 19時半で
店: ご予約のお名前をお願いします。
客: 佐藤と申します
店: 11 月 27 日 (金) 19:30 から 2 名様、佐藤様でご予約を承りました。
{'日付': '2026-11-27', '時刻': '19:30', '人数': 2, '名前': '佐藤'} 



- 対話の状態 (どの項目が埋まったか) をプログラムで管理し、足りないものを順に聞く。フレームに基づく対話管理と呼ばれる、タスク指向対話の基本形。
- 聞いた項目だけでなく、発話に含まれる情報はすべて取り込む。「明日の 7 時に 4 人」と言った客に「何名様ですか」と聞き返さないため。
- 実用では、確定前の復唱と訂正 (「19 時ではなく 20 時で」) の扱い、営業時間外や満席の確認が必要になる。
- 抽出を LLM に任せて (JSON で出力させて)、状態管理と最終確認だけを規則で行う構成も多い。予約や決済のように間違えられない処理は、LLM に直接やらせない。